# Randomized PCA

## Co je Randomized PCA?

Randomized PCA (Principal Component Analysis) je efektivní varianta klasického algoritmu PCA, která je optimalizovaná pro velké datové sady s vysokou dimenzionalitou. Na rozdíl od standardního PCA, který počítá úplný rozklad SVD (Singular Value Decomposition), Randomized PCA používá aproximativní přístup založený na náhodných projekcích, čímž výrazně snižuje výpočetní náročnost.

### Klíčové vlastnosti Randomized PCA:

- **Výpočetní efektivita**: Výrazně rychlejší než standardní PCA pro velké datové sady
- **Paměťová efektivita**: Vyžaduje méně paměti než tradiční PCA
- **Přesnost aproximace**: Poskytuje dobrou aproximaci hlavních komponent s kontrolovatelnou přesností
- **Škálovatelnost**: Vhodná pro datové sady s tisíci až miliony dimenzí

### Kdy použít Randomized PCA:

1. **Velké datové sady**: Když pracujete s daty s mnoha příznaky (vysoká dimenzionalita)
2. **Výpočetní omezení**: Když máte omezené výpočetní prostředky nebo potřebujete rychlé výsledky
3. **Průzkumná analýza dat**: Pro rychlou vizualizaci a průzkum struktury dat
4. **Předzpracování dat**: Jako krok předzpracování pro další algoritmy strojového učení

V scikit-learn je Randomized PCA implementován v rámci klasické třídy `PCA`, kde se aktivuje nastavením parametru `svd_solver='randomized'`.

In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits, fetch_olivetti_faces, load_iris, make_blobs
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import time
import warnings

# Pro lepší vizualizaci
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
warnings.filterwarnings('ignore')
np.random.seed(42)

## 1. Základy Randomized PCA

Nejprve ukážeme základní použití Randomized PCA a porovnáme jej s klasickým PCA na jednoduchých datech.

In [ ]:
# Vytvoříme jednoduchá syntetická data
n_samples = 1000
n_features = 500  # Relativně vysoký počet příznaků

# Vygenerujeme data s jasnou strukturou ve 3 dimenzích, zbytek je šum
# První 3 příznaky mají větší rozptyl, ostatní jsou malý šum
X = np.random.randn(n_samples, n_features)
X[:, 0] *= 10  # První příznak má největší rozptyl
X[:, 1] *= 5   # Druhý příznak má střední rozptyl
X[:, 2] *= 2   # Třetí příznak má menší rozptyl

print(f"Tvar dat: {X.shape}")
print(f"Průměrná hodnota: {X.mean():.3f}")
print(f"Směrodatná odchylka: {X.std():.3f}")

In [ ]:
# Porovnání rychlosti standardního PCA a Randomized PCA
n_components = 10  # Chceme zachovat 10 hlavních komponent

# Standardní PCA
start_time = time.time()
pca_standard = PCA(n_components=n_components, svd_solver='full')
X_standard = pca_standard.fit_transform(X)
standard_time = time.time() - start_time

# Randomized PCA
start_time = time.time()
pca_randomized = PCA(n_components=n_components, svd_solver='randomized', random_state=42)
X_randomized = pca_randomized.fit_transform(X)
randomized_time = time.time() - start_time

print(f"Standardní PCA: {standard_time:.4f} sekund")
print(f"Randomized PCA: {randomized_time:.4f} sekund")
print(f"Zrychlení: {standard_time / randomized_time:.2f}x")

In [ ]:
# Porovnejme vysvětlený rozptyl z obou metod
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(range(1, n_components + 1), pca_standard.explained_variance_ratio_, alpha=0.8, label='Standardní PCA')
plt.bar(range(1, n_components + 1), pca_randomized.explained_variance_ratio_, alpha=0.5, label='Randomized PCA')
plt.xlabel('Komponenta')
plt.ylabel('Podíl vysvětleného rozptylu')
plt.legend()
plt.title('Porovnání vysvětleného rozptylu')

plt.subplot(1, 2, 2)
plt.plot(np.cumsum(pca_standard.explained_variance_ratio_), marker='o', linestyle='-', label='Standardní PCA')
plt.plot(np.cumsum(pca_randomized.explained_variance_ratio_), marker='x', linestyle='--', label='Randomized PCA')
plt.xlabel('Počet komponent')
plt.ylabel('Kumulativní vysvětlený rozptyl')
plt.legend()
plt.title('Kumulativní vysvětlený rozptyl')

plt.tight_layout()
plt.show()

# Číselné porovnání
print("Podíl vysvětleného rozptylu první komponenty:")
print(f"Standardní PCA: {pca_standard.explained_variance_ratio_[0]:.4f}")
print(f"Randomized PCA: {pca_randomized.explained_variance_ratio_[0]:.4f}")

print("\nCelkový vysvětlený rozptyl (10 komponent):")
print(f"Standardní PCA: {np.sum(pca_standard.explained_variance_ratio_):.4f}")
print(f"Randomized PCA: {np.sum(pca_randomized.explained_variance_ratio_):.4f}")

In [ ]:
# Vizualizujme data v prvních dvou komponentách
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_standard[:, 0], X_standard[:, 1], alpha=0.7, s=10)
plt.title('Standardní PCA')
plt.xlabel('První hlavní komponenta')
plt.ylabel('Druhá hlavní komponenta')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_randomized[:, 0], X_randomized[:, 1], alpha=0.7, s=10, c='orange')
plt.title('Randomized PCA')
plt.xlabel('První hlavní komponenta')
plt.ylabel('Druhá hlavní komponenta')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Vliv parametru `n_components` a `random_state`

Prozkoumejme, jak parametr `n_components` a `random_state` ovlivňují výsledky Randomized PCA.

In [ ]:
# Vliv počtu komponent na vysvětlený rozptyl a výpočetní čas
component_range = [2, 5, 10, 20, 50, 100]
times_randomized = []
times_standard = []
var_randomized = []
var_standard = []

for n_comp in component_range:
    # Standardní PCA
    start_time = time.time()
    pca_std = PCA(n_components=n_comp, svd_solver='full')
    pca_std.fit(X)
    times_standard.append(time.time() - start_time)
    var_standard.append(np.sum(pca_std.explained_variance_ratio_))
    
    # Randomized PCA
    start_time = time.time()
    pca_rand = PCA(n_components=n_comp, svd_solver='randomized', random_state=42)
    pca_rand.fit(X)
    times_randomized.append(time.time() - start_time)
    var_randomized.append(np.sum(pca_rand.explained_variance_ratio_))

# Vizualizace výsledků
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(component_range, times_standard, marker='o', linestyle='-', label='Standardní PCA')
plt.plot(component_range, times_randomized, marker='x', linestyle='--', label='Randomized PCA')
plt.xlabel('Počet komponent')
plt.ylabel('Výpočetní čas [s]')
plt.legend()
plt.title('Výpočetní čas vs. počet komponent')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(component_range, var_standard, marker='o', linestyle='-', label='Standardní PCA')
plt.plot(component_range, var_randomized, marker='x', linestyle='--', label='Randomized PCA')
plt.xlabel('Počet komponent')
plt.ylabel('Celkový vysvětlený rozptyl')
plt.legend()
plt.title('Vysvětlený rozptyl vs. počet komponent')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Tabulka s výsledky
results = pd.DataFrame({
    'Počet komponent': component_range,
    'Čas - Standardní PCA [s]': times_standard,
    'Čas - Randomized PCA [s]': times_randomized,
    'Zrychlení': np.array(times_standard) / np.array(times_randomized),
    'Vysvětlený rozptyl - Standardní PCA': var_standard,
    'Vysvětlený rozptyl - Randomized PCA': var_randomized,
    'Rozdíl v rozptylu [%]': ((np.array(var_standard) - np.array(var_randomized)) / np.array(var_standard)) * 100
})

print("Porovnání Standardní PCA vs. Randomized PCA:")
print(results.round(4))

In [ ]:
# Vliv random_state na stabilitu výsledků
random_states = [42, 123, 999, 2023, 7]
n_comp = 10

# Vypočítáme rozptyl vysvětlený prvními 3 komponentami pro různé random_state
var_results = []

for rs in random_states:
    pca_rand = PCA(n_components=n_comp, svd_solver='randomized', random_state=rs)
    pca_rand.fit(X)
    var_results.append(pca_rand.explained_variance_ratio_[:3])

var_results = np.array(var_results)

# Vizualizace stability
plt.figure(figsize=(10, 6))

# Pro každou z prvních 3 komponent
for i in range(3):
    plt.plot(random_states, var_results[:, i], marker='o', label=f'Komponenta {i+1}')

plt.xlabel('Random state')
plt.ylabel('Vysvětlený rozptyl')
plt.title('Stabilita vysvětleného rozptylu pro různé hodnoty random_state')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Spočítáme statistiky stability
mean_var = np.mean(var_results, axis=0)
std_var = np.std(var_results, axis=0)
cv_var = std_var / mean_var * 100  # Koeficient variace v procentech

stability_stats = pd.DataFrame({
    'Komponenta': [1, 2, 3],
    'Průměrný vysvětlený rozptyl': mean_var,
    'Směrodatná odchylka': std_var,
    'Variační koeficient [%]': cv_var
})

print("Stabilita Randomized PCA pro různé random_state:")
print(stability_stats.round(4))

## 3. Aplikace Randomized PCA na reálná data - MNIST Digits

Podívejme se na použití Randomized PCA na reálném datasetu - MNIST digits (ručně psané číslice).

In [ ]:
# Načtení datasetu MNIST digits
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Tvar dat: {X_digits.shape}")
print(f"Počet tříd: {len(np.unique(y_digits))}")

# Zobrazení několika příkladů
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Digit: {y_digits[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Aplikace Standardní PCA vs Randomized PCA
n_components_digits = 20  # Redukujeme z 64 na 20 dimenzí

# Standardní PCA
start_time = time.time()
pca_std_digits = PCA(n_components=n_components_digits, svd_solver='full')
X_std_digits = pca_std_digits.fit_transform(X_digits)
std_time_digits = time.time() - start_time

# Randomized PCA
start_time = time.time()
pca_rand_digits = PCA(n_components=n_components_digits, svd_solver='randomized', random_state=42)
X_rand_digits = pca_rand_digits.fit_transform(X_digits)
rand_time_digits = time.time() - start_time

print(f"Standardní PCA čas: {std_time_digits:.4f} sekund")
print(f"Randomized PCA čas: {rand_time_digits:.4f} sekund")
print(f"Zrychlení: {std_time_digits / rand_time_digits:.2f}x")
print(f"\nVysvětlený rozptyl (20 komponent):")
print(f"Standardní PCA: {np.sum(pca_std_digits.explained_variance_ratio_):.4f}")
print(f"Randomized PCA: {np.sum(pca_rand_digits.explained_variance_ratio_):.4f}")

In [ ]:
# Vizualizace dat ve 2D pomocí prvních dvou komponent
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
scatter = plt.scatter(X_std_digits[:, 0], X_std_digits[:, 1], c=y_digits, cmap='tab10', 
                     alpha=0.8, s=10, edgecolor='none')
plt.colorbar(label='Číslice')
plt.title('Standardní PCA - MNIST Digits')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
scatter = plt.scatter(X_rand_digits[:, 0], X_rand_digits[:, 1], c=y_digits, cmap='tab10', 
                     alpha=0.8, s=10, edgecolor='none')
plt.colorbar(label='Číslice')
plt.title('Randomized PCA - MNIST Digits')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Podívejme se na vlastní vektory (eigenvectors) - první komponenty
# Tyto představují "prototypické" vzory v datech
plt.figure(figsize=(12, 3))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(pca_rand_digits.components_[i].reshape(8, 8), cmap='viridis')
    plt.title(f'Komponenta {i+1}')
    plt.axis('off')
plt.suptitle('První komponenty (vlastní vektory) Randomized PCA', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Rekonstrukce původních obrázků z redukovaných dat
# Porovnáme originály a rekonstruované obrázky pro různý počet komponent
digits_sample = X_digits[10:15]  # Vybereme několik čísel pro porovnání
n_components_list = [5, 10, 20, 30]

plt.figure(figsize=(15, 10))

# Zobrazit originální obrázky v prvním řádku
for i in range(5):
    plt.subplot(len(n_components_list) + 1, 5, i + 1)
    plt.imshow(digits_sample[i].reshape(8, 8), cmap='gray')
    if i == 2:
        plt.title('Originální obrázky', fontsize=14)
    plt.axis('off')

# Pro každý počet komponent rekonstruujeme obrázky
for j, n_comp in enumerate(n_components_list):
    # Redukce a zpětná rekonstrukce
    pca = PCA(n_components=n_comp, svd_solver='randomized', random_state=42)
    reduced = pca.fit_transform(X_digits)
    reconstructed = pca.inverse_transform(reduced)
    
    # Zobrazení rekonstruovaných obrázků
    for i in range(5):
        plt.subplot(len(n_components_list) + 1, 5, (j + 1) * 5 + i + 1)
        plt.imshow(reconstructed[10 + i].reshape(8, 8), cmap='gray')
        if i == 0:
            plt.ylabel(f'{n_comp} komponent', fontsize=12)
        plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Využití PCA pro klasifikaci - porovnáme výkon modelu s a bez redukce dimenzí
# Rozdělíme data na trénovací a testovací sadu
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=42)

# Funkce pro trénování a vyhodnocení modelu
def evaluate_model(X_train, X_test, y_train, y_test, model_name):
    start_time = time.time()
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    start_time = time.time()
    y_pred = clf.predict(X_test)
    pred_time = time.time() - start_time
    
    acc = accuracy_score(y_test, y_pred)
    
    print(f"{model_name}:")
    print(f"  Přesnost: {acc:.4f}")
    print(f"  Doba trénování: {train_time:.4f} s")
    print(f"  Doba predikce: {pred_time:.4f} s")
    return acc, train_time, pred_time

# Model bez redukce dimenzionality
print("Model na původních datech:")
acc_orig, train_orig, pred_orig = evaluate_model(X_train, X_test, y_train, y_test, "Původní data (64 příznaků)")

# Model s redukcí dimenzionality pomocí Randomized PCA
component_counts = [5, 10, 20, 30, 40]
results = []

for n_comp in component_counts:
    pca = PCA(n_components=n_comp, svd_solver='randomized', random_state=42)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    
    print(f"\nModel s Randomized PCA ({n_comp} komponent):")
    acc, train_time, pred_time = evaluate_model(X_train_pca, X_test_pca, y_train, y_test, 
                                              f"Randomized PCA ({n_comp} komponent)")
    
    results.append({
        'Počet komponent': n_comp,
        'Přesnost': acc,
        'Doba trénování': train_time,
        'Doba predikce': pred_time,
        'Zrychlení trénování': train_orig / train_time,
        'Zrychlení predikce': pred_orig / pred_time
    })
    
# Vizualizace výsledků
results_df = pd.DataFrame(results)
print("\nShrnutí výsledků:")
print(results_df.round(4))

In [ ]:
# Graf přesnosti vs. počet komponent
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(results_df['Počet komponent'], results_df['Přesnost'], 'o-', linewidth=2)
plt.axhline(y=acc_orig, color='r', linestyle='--', label=f'Původní data (přesnost: {acc_orig:.4f})')
plt.xlabel('Počet komponent')
plt.ylabel('Přesnost')
plt.title('Přesnost vs. počet komponent')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(results_df['Počet komponent'], results_df['Doba trénování'], 'o-', linewidth=2, label='Doba trénování')
plt.plot(results_df['Počet komponent'], results_df['Doba predikce'], 'x-', linewidth=2, label='Doba predikce')
plt.axhline(y=train_orig, color='r', linestyle='--', label=f'Původní data (trénování)')
plt.axhline(y=pred_orig, color='r', linestyle=':', label=f'Původní data (predikce)')
plt.xlabel('Počet komponent')
plt.ylabel('Doba [s]')
plt.title('Výpočetní čas vs. počet komponent')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## 4. Škálování na větší data - Obličejová data

Podívejme se na výkon Randomized PCA na větším datasetu obličejových fotografií.

In [ ]:
# Načtení datasetu obličejů
try:
    faces = fetch_olivetti_faces()
    X_faces = faces.data
    y_faces = faces.target
    
    print(f"Tvar dat: {X_faces.shape}")
    print(f"Počet tříd (osob): {len(np.unique(y_faces))}")
    
    # Zobrazení několika příkladů
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(faces.images[i], cmap='gray')
        ax.set_title(f'Osoba: {y_faces[i]}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Porovnání rychlosti PCA na větším datasetu
    n_components_faces = 50
    
    # Standardní PCA
    start_time = time.time()
    pca_std_faces = PCA(n_components=n_components_faces, svd_solver='full')
    X_std_faces = pca_std_faces.fit_transform(X_faces)
    std_time_faces = time.time() - start_time
    
    # Randomized PCA
    start_time = time.time()
    pca_rand_faces = PCA(n_components=n_components_faces, svd_solver='randomized', random_state=42)
    X_rand_faces = pca_rand_faces.fit_transform(X_faces)
    rand_time_faces = time.time() - start_time
    
    print(f"Standardní PCA čas: {std_time_faces:.4f} sekund")
    print(f"Randomized PCA čas: {rand_time_faces:.4f} sekund")
    print(f"Zrychlení: {std_time_faces / rand_time_faces:.2f}x")
    print(f"\nVysvětlený rozptyl (50 komponent):")
    print(f"Standardní PCA: {np.sum(pca_std_faces.explained_variance_ratio_):.4f}")
    print(f"Randomized PCA: {np.sum(pca_rand_faces.explained_variance_ratio_):.4f}")
    
    # Zobrazení tzv. "eigenfaces" - vlastní vektorů
    plt.figure(figsize=(12, 3))
    for i in range(5):
        plt.subplot(1, 5, i+1)
        plt.imshow(pca_rand_faces.components_[i].reshape(64, 64), cmap='gray')
        plt.title(f'Eigenface {i+1}')
        plt.axis('off')
    plt.suptitle('První komponenty Randomized PCA (eigenfaces)', y=1.05)
    plt.tight_layout()
    plt.show()
    
    # 2D zobrazení obličejů
    plt.figure(figsize=(10, 8))
    plt.scatter(X_rand_faces[:, 0], X_rand_faces[:, 1], c=y_faces, cmap='tab20', 
               alpha=0.8, s=30, edgecolor='k')
    plt.colorbar(label='Osoba ID')
    plt.title('2D projekce obličejů pomocí Randomized PCA', fontsize=14)
    plt.xlabel('PC1', fontsize=12)
    plt.ylabel('PC2', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Nelze načíst dataset obličejů: {e}")
    print("Přeskakuji tuto sekci...")

## 5. Porovnání Randomized PCA s jinými metodami redukce dimenzionality

Konečně porovnejme Randomized PCA s několika dalšími metodami redukce dimenzionality.

In [ ]:
# Pro ukázku použijeme datasets Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print(f"Dataset Iris: {X_iris.shape}")

# Import dalších metod redukce dimenzionality
from sklearn.manifold import TSNE, Isomap
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Standardizace dat
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Čas a výsledky pro každou metodu
methods = {}

# 1. Standardní PCA
start_time = time.time()
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris_scaled)
methods['PCA'] = {
    'time': time.time() - start_time,
    'data': X_pca,
    'var_explained': np.sum(pca.explained_variance_ratio_)
}

# 2. Randomized PCA
start_time = time.time()
rpca = PCA(n_components=2, svd_solver='randomized', random_state=42)
X_rpca = rpca.fit_transform(X_iris_scaled)
methods['Randomized PCA'] = {
    'time': time.time() - start_time,
    'data': X_rpca,
    'var_explained': np.sum(rpca.explained_variance_ratio_)
}

# 3. t-SNE
start_time = time.time()
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_iris_scaled)
methods['t-SNE'] = {
    'time': time.time() - start_time,
    'data': X_tsne,
}

# 4. Isomap
start_time = time.time()
isomap = Isomap(n_components=2, n_neighbors=7)
X_isomap = isomap.fit_transform(X_iris_scaled)
methods['Isomap'] = {
    'time': time.time() - start_time,
    'data': X_isomap,
}

# 5. LDA
start_time = time.time()
lda = LDA(n_components=2)
X_lda = lda.fit_transform(X_iris_scaled, y_iris)
methods['LDA'] = {
    'time': time.time() - start_time,
    'data': X_lda,
}

# Vizualizace všech metod
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (method_name, method_data) in enumerate(methods.items()):
    ax = axes[i]
    X_transformed = method_data['data']
    ax.scatter(X_transformed[:, 0], X_transformed[:, 1], c=y_iris, cmap='viridis',
              alpha=0.8, s=40, edgecolor='k')
    ax.set_title(f'{method_name}\nČas: {method_data["time"]:.4f}s', fontsize=12)
    if 'var_explained' in method_data:
        ax.set_xlabel(f'Vysvětlený rozptyl: {method_data["var_explained"]:.3f}', fontsize=10)
    ax.grid(True, alpha=0.3)

# Skryjeme prázdný subplot
axes[-1].axis('off')
    
plt.suptitle('Porovnání metod redukce dimenzionality na Iris datasetu', fontsize=16, y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

## 6. Shrnutí a doporučení

### Výhody Randomized PCA:

1. **Výpočetní efektivita**: Výrazně rychlejší než standardní PCA pro velké datové sady
2. **Paměťová efektivita**: Vyžaduje méně paměti pro zpracování vysokodimenzionálních dat
3. **Téměř identické výsledky**: Poskytuje velmi podobné výsledky jako standardní PCA s minimálním rozdílem v přesnosti
4. **Škálovatelnost**: Vhodný pro datové sady s tisíci či miliony dimenzí

### Nevýhody Randomized PCA:

1. **Mírná ztráta přesnosti**: Pro stejný počet komponent může vysvětlit mírně méně rozptylu než standardní PCA
2. **Závislost na náhodném stavu**: Výsledky se mohou mírně lišit podle nastaveného random_state
3. **Méně vhodný pro malé datové sady**: Pro malé datové sady nepřináší výrazné zrychlení a standardní PCA může být lepší volbou

### Doporučení pro použití:

- **Kdy použít Randomized PCA**:
  - Při práci s datovými sadami s více než ~1000 příznaky
  - Když je výpočetní čas kritickým faktorem
  - V pipeline zpracování dat, kde je redukce dimenzionality jen mezikrokem
  - Pro rychlou průzkumnou analýzu dat a vizualizaci

- **Kdy použít standardní PCA**:
  - Při práci s menšími datovými sadami
  - Když je maximální přesnost kritičtější než rychlost
  - Když potřebujete konzistentní výsledky bez závislosti na náhodném stavu

- **Parametry k ladění**:
  - `n_components`: Počet dimenzí k zachování (experimentujte s různými hodnotami)
  - `random_state`: Pro zajištění reprodukovatelnosti
  - `iterated_power`: Pro zvýšení přesnosti aproximace (vyšší hodnoty → větší přesnost, ale delší výpočet)

V praxi je Randomized PCA často výchozí volbou pro většinu úloh redukce dimenzionality na velkých datových sadách, kde nabízí vynikající kompromis mezi rychlostí a přesností.